# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema, accessible at its source URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Accessing dataset metadata object attributes with dot notation
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Examine the dataset for available record sets (`@id`), their fields, and columns. All fields are referenced by their `@id`.

In [ ]:
# List all available record sets by @id
print("Record sets available in metadata (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else 'n/a'}")
    print("  Fields and columns:")
    for f in getattr(rs, 'fields', []):
        print(f"    Field @id: {f.id}")
        print(f"      name: {f.name}")
        if hasattr(f, 'column'):
            col = f.column
            if hasattr(col, 'id'):
                print(f"      column @id: {col.id}")
    print()

# For later extraction, collect record set @id(s)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

## 3. Data Extraction
Load data from a specific record set into a DataFrame, referencing by record set and field `@id` as per the schema overview above.

In [ ]:
# Example: extract all record sets to pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet {rs_id} --> DataFrame shape: {df.shape}")
    except Exception as e:
        print(f"Could not load RecordSet {rs_id}: {e}")

# Select first record set for illustration
main_record_set_id = record_set_ids[0]
print(f"\nColumns in main DataFrame (from RecordSet {main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Select a numeric field for basic filtering, normalization and group analysis. All operations use field `@id`s for reference.

In [ ]:
# Choose a numeric field and optionally a group field by @id
# For demonstration, we scan numeric fields in the main record set
main_df = dataframes[main_record_set_id]

numeric_field_id = None
group_field_id = None
# Try to infer numeric and group fields by dtype
for col in main_df.columns:
    if pd.api.types.is_numeric_dtype(main_df[col]) and numeric_field_id is None:
        numeric_field_id = col
    if group_field_id is None and pd.api.types.is_string_dtype(main_df[col]):
        group_field_id = col
print(f"Selected numeric_field_id: {numeric_field_id}")
print(f"Selected group_field_id: {group_field_id}")

if numeric_field_id is not None:
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize selected numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} values:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by group_field if it exists
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize the numeric distribution and grouped means using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

    if group_field_id and group_field_id in main_df.columns:
        grouped = main_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()


## 6. Conclusion
In this notebook, we loaded the FAIR^2 mass spectrometry dataset via its Croissant schema, explored its schema and record sets, extracted data using entity `@id` references, performed basic exploratory data analysis and visualized the results. 

- All fields and entities are referenced by their unique Croissant `@id`.
- The `mlcroissant` library allows semantic, robust loading and processing of FAIR datasets.
- Users can further customize data cleaning, transformation, or modeling workflows by adjusting the extraction and analysis steps.

> For deeper exploration, consult dataset documentation and consider the biological interpretations of quantitative protein data contained.